# 🎙️ LifeMate v3.0 — High-Performance Voice-to-Action Agent Pipeline
### Zalo AI Challenge 2026

This notebook implements the complete **v3.0 Top-Tier Pipeline**:
1. **Audio Tai Nghe**: `openai/whisper-large-v3-turbo` (809M params, ultra-accurate, zero-bias transcription with volume normalization).
2. **Agent Não Bộ**: Supervised Fine-Tuning (LoRA r=32, alpha=64, 8192 tokens) on `Qwen/Qwen3-4B-Instruct-2507` (4.0B < 5B constraint).
3. **Loss Masking**: Completion-Only Loss Collator ensuring 100% gradient updates on Assistant tool calls.
4. **Auto-Merge**: Merges LoRA weights into a standalone offline model.
5. **Inference & Validation**: Runs interactive `zaic_tools` agent on `public_test` with entity resolution.

In [ ]:
# 1. GPU Check
!nvidia-smi

In [ ]:
# 2. Install Core Dependencies (Transformers >= 4.51.0 required for Qwen3-4B)
!pip install -q -U "transformers>=4.51.0" soundfile accelerate peft trl datasets
!pip uninstall -y torchao

# Install NVIDIA NeMo for Parakeet ASR
!pip install -q nemo_toolkit[asr]

In [ ]:
# 3. Clone Repository
import os
REPO_URL = "https://github.com/Quandinh030106/LifeMate-Voice2Action-Agent.git"
if not os.path.exists("/kaggle/working/LifeMate-Voice2Action-Agent"):
    !git clone $REPO_URL /kaggle/working/LifeMate-Voice2Action-Agent
else:
    !cd /kaggle/working/LifeMate-Voice2Action-Agent && git fetch origin && git reset --hard origin/main

%cd /kaggle/working/LifeMate-Voice2Action-Agent

In [ ]:
# 4. Locate Dataset Directory
import glob, os
datasets = glob.glob("/kaggle/input/**/public_test.jsonl", recursive=True)
if datasets:
    DATA_DIR = os.path.dirname(os.path.dirname(datasets[0]))
    print("Found Data Directory:", DATA_DIR)
else:
    DATA_DIR = "/kaggle/input/lifemate-data/train+public_test/train+public_test"
    print("Defaulting to:", DATA_DIR)

os.environ["DATA_DIR"] = DATA_DIR


In [ ]:
# 5. Step 1: Prepare SFT Training Data from train.jsonl (1545 multi-turn samples)
!python -u scripts/prepare_sft_data.py \
    --data_dir "$DATA_DIR" \
    --output_dir data/sft

In [ ]:
# 6. Step 2: Fine-Tune Qwen2.5-3B-Instruct via LoRA and Merge
import subprocess, sys, os, glob, zipfile

print('=' * 60)
print('🔍 LifeMate Model Detector: Searching for Pre-Trained Weights...')
print('=' * 60)

candidate_models = glob.glob('/kaggle/input/**/lifemate-qwen-sft', recursive=True)
candidate_adapter_dirs = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)]
all_zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
candidate_zips = [z for z in all_zips if 'lora' in z.lower() or 'adapter' in z.lower() or ('data' not in z.lower() and 'train' not in z.lower() and 'public' not in z.lower())]

if os.path.exists('/kaggle/working/lifemate-qwen-sft') and os.listdir('/kaggle/working/lifemate-qwen-sft'):
    MODEL_DIR = '/kaggle/working/lifemate-qwen-sft'
    print(f'✅ Found existing merged model in working dir: {MODEL_DIR}')
    print('⏩ Skipping training completely!')
    SKIP_TRAIN = True
elif candidate_models:
    MODEL_DIR = candidate_models[0]
    print(f'✅ Found pre-trained merged model in Input: {MODEL_DIR}')
    print('⏩ Skipping training completely!')
    SKIP_TRAIN = True
elif candidate_adapter_dirs:
    adapter_dir = candidate_adapter_dirs[0]
    print(f'✅ Found unzipped LoRA adapter in Input: {adapter_dir}')
    print('⚡ Merging LoRA into base Qwen2.5-3B-Instruct on CPU (takes ~30s)...')
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch, gc
    base_model_name = 'Qwen/Qwen3-4B-Instruct-2507'
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        device_map='cpu'
    )
    peft_model = PeftModel.from_pretrained(base_model, adapter_dir)
    merged_model = peft_model.merge_and_unload()
    MODEL_DIR = '/kaggle/working/lifemate-qwen-sft'
    os.makedirs(MODEL_DIR, exist_ok=True)
    merged_model.save_pretrained(MODEL_DIR)
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    tokenizer.save_pretrained(MODEL_DIR)
    del base_model, peft_model, merged_model
    gc.collect()
    print(f'🎉 Merged model ready at {MODEL_DIR}!')
    print('⏩ Skipping training completely!')
    SKIP_TRAIN = True
elif candidate_zips:
    zip_path = candidate_zips[0]
    print(f'✅ Found LoRA adapter zip file in Input: {zip_path}')
    print('📦 Unzipping LoRA adapter...')
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch, gc
    unzip_dir = '/kaggle/working/qwen-lora'
    os.makedirs(unzip_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(unzip_dir)
    peft_dir = unzip_dir
    if not os.path.exists(os.path.join(peft_dir, 'adapter_config.json')):
        sub_configs = glob.glob(f'{unzip_dir}/**/adapter_config.json', recursive=True)
        if sub_configs:
            peft_dir = os.path.dirname(sub_configs[0])
    print(f'⚡ Merging LoRA from {peft_dir} into base Qwen2.5-3B-Instruct on CPU (takes ~30s)...')
    base_model_name = 'Qwen/Qwen3-4B-Instruct-2507'
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        device_map='cpu'
    )
    peft_model = PeftModel.from_pretrained(base_model, peft_dir)
    merged_model = peft_model.merge_and_unload()
    MODEL_DIR = '/kaggle/working/lifemate-qwen-sft'
    os.makedirs(MODEL_DIR, exist_ok=True)
    merged_model.save_pretrained(MODEL_DIR)
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    tokenizer.save_pretrained(MODEL_DIR)
    del base_model, peft_model, merged_model
    gc.collect()
    print(f'🎉 Merged model ready at {MODEL_DIR}!')
    print('⏩ Skipping training completely!')
    SKIP_TRAIN = True
else:
    MODEL_DIR = '/kaggle/working/lifemate-qwen-sft'
    print('ℹ️ No pre-trained model or LoRA adapter found in Input or Working.')
    print('🚀 Starting fresh SFT Training on 1,545 multi-turn samples (3 epochs, ~1.5h)...')
    SKIP_TRAIN = False

os.environ['MODEL_DIR'] = MODEL_DIR

if not SKIP_TRAIN:
    cmd = [
        sys.executable, '-u', 'train_sft.py',
        '--model_name_or_path', 'Qwen/Qwen3-4B-Instruct-2507',
        '--train_file', 'data/sft/sft_train.jsonl',
        '--val_file', 'data/sft/sft_val.jsonl',
        '--output_dir', '/kaggle/working/qwen-lora',
        '--merged_dir', '/kaggle/working/lifemate-qwen-sft',
        '--epochs', '3',
        '--batch_size', '1',
        '--grad_accum', '8',
        '--lr', '1.5e-4',
        '--lora_r', '32',
        '--lora_alpha', '64',
        '--max_seq_length', '8192'
    ]
    print('Executing:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print(f'✅ Ready to run inference using model at {MODEL_DIR}')


In [ ]:
# 7. Step 3: Run Full Inference on public_test with Two-Stage Pipeline (~10-15 mins)
import os
# Ensure incomplete submission from any previous failed run is removed to regenerate cleanly
sub_path = '/kaggle/working/submission.jsonl'
if os.path.exists(sub_path):
    with open(sub_path, 'r', encoding='utf-8') as f_sub:
        count = sum(1 for line in f_sub if line.strip())
    if count < 634:
        print(f'Removing incomplete submission ({count}/634 lines) to regenerate from sample 1 cleanly...')
        os.remove(sub_path)
    else:
        print(f'Submission already complete ({count}/634 lines). Running validator...')

!python -u inference.py \
    --data_dir "$DATA_DIR" \
    --split public_test \
    --asr_engine openai \
    --asr_model openai/whisper-large-v3-turbo \
    --llm_model "$MODEL_DIR" \
    --output /kaggle/working/submission.jsonl


In [ ]:
# 8. Step 4: Clean, Deduplicate, and Validate Output with Official zaic_tools Validator
!python scripts/clean_submission.py --input /kaggle/working/submission.jsonl --data_dir "$DATA_DIR"

import os, subprocess
env = os.environ.copy()
env["PYTHONPATH"] = f"{DATA_DIR}/public_test:" + env.get("PYTHONPATH", "")
cmd = [
    "python", "-m", "zaic_tools",
    "--split", "public_test",
    "--root", DATA_DIR,
    "--validate", "/kaggle/working/submission.jsonl"
]
print("Running official zaic_tools validator...")
res = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print(res.stderr)


In [ ]:
# 9. Step 5: Check submission line count and preview
!wc -l /kaggle/working/submission.jsonl
!head -n 5 /kaggle/working/submission.jsonl

In [ ]:
# 10. Step 6: Backup LoRA Adapter for Future Runs (~115MB only)
import os, shutil
adapter_dir = '/kaggle/working/qwen-lora'
if os.path.exists(adapter_dir):
    shutil.make_archive('/kaggle/working/qwen_lora_adapter', 'zip', adapter_dir)
    print('LoRA adapter archived successfully at /kaggle/working/qwen_lora_adapter.zip')
    !ls -lh /kaggle/working/qwen_lora_adapter.zip
else:
    print('Adapter directory not found (pre-trained model was used directly).')
